# MinerU LayoutExtractor Test Notebook

This notebook demonstrates and tests the **MinerUExtractor** strategy (an alternative Strategy B)
which uses **MinerU** for layout-aware extraction via a normalised JSON schema:

- Table-heavy and mixed-layout documents
- OCR-heavy or scanned PDFs (when MinerU has been run with OCR)
- Normalisation into the shared `ExtractedDocument` schema via `MinerUAdapter`.

> Prerequisites:
> - MinerU installed, e.g. `uv pip install -U "mineru[all]"`
> - MinerU has already been run on the chosen PDF, producing a JSON file
>   (e.g. `<pdf>.mineru.json`) matching the schema expected by `MinerUAdapter`.


In [ ]:
# Setup and Imports

import sys
from pathlib import Path

# Ensure src is on the path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies import MinerUExtractor
from src.models.document_profile import DocumentProfile


In [ ]:
# Select Test Document and MinerU JSON

DATA_ROOT = Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data")

TEST_DOCS = {
    "class_a": DATA_ROOT / "class_a" / "CBE_ANNUAL_REPORT_2023-24.pdf",
    "class_b": DATA_ROOT / "class_b" / "Annual_Report_JUNE-2023.pdf",
    "class_c": DATA_ROOT / "class_c" / "fta_performance_survey_final_report_2022.pdf",
    "class_d": DATA_ROOT / "class_d" / "tax_expenditure_ethiopia_2021_22.pdf",
}

selected_doc = "class_c"  # change as needed
pdf_path = TEST_DOCS[selected_doc]

# Default MinerU JSON path convention: <pdf>.mineru.json
mineru_json_path = pdf_path.with_suffix(pdf_path.suffix + ".mineru.json")

print(f"PDF: {pdf_path}")
print(f"Expected MinerU JSON: {mineru_json_path}")
print(f"PDF exists: {pdf_path.exists()}")
print(f"MinerU JSON exists: {mineru_json_path.exists()}")


In [ ]:
# Step 1: Triage and Profile

profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

print("🔍 Running Triage Agent...")
profile: DocumentProfile = triage.classify_document(pdf_path)

print("\n📋 Document Profile (summary):")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print(f"  - Estimated Cost: {profile.estimated_cost}")
print(f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  - Pages: {profile.metadata.page_count}")


In [ ]:
# Step 2: Initialise MinerUExtractor and Run Extraction

if not mineru_json_path.exists():
    raise FileNotFoundError(
        f"MinerU JSON not found at {mineru_json_path}.\n"
        "Run MinerU on this PDF first (CLI or API) and emit a JSON file "
        "that matches the schema expected by `MinerUAdapter`."
    )

mineru_extractor = MinerUExtractor(mineru_json_path=mineru_json_path)

print("📄 Extracting with MinerUExtractor...")

import time
start_time = time.time()

extracted = mineru_extractor.extract(str(pdf_path))

elapsed = time.time() - start_time
print(f"\n✓ Extraction completed in {elapsed:.2f} seconds")
print("\n📊 Extraction Summary (MinerU → ExtractedDocument):")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")
print(f"  - Reading Order indices: {len(extracted.reading_order)}")


In [ ]:
# Step 3: Inspect Sample Tables and Figures

print("📊 Sample Tables (first 3):")
print("=" * 80)
for i, table in enumerate(extracted.tables[:3], start=1):
    print(f"\n[Table {i}] Page {table.page_num}")
    print(f"  Headers: {table.headers}")
    if table.rows:
        print(f"  First row: {table.rows[0]}")

print("\n🖼️ Sample Figures (first 3):")
print("=" * 80)
for i, fig in enumerate(extracted.figures[:3], start=1):
    print(f"\n[Figure {i}] Page {fig.page_num}")
    print(f"  Caption: {fig.caption or '(none)'}")
    print(f"  BBox: ({fig.bbox.x0:.1f}, {fig.bbox.y0:.1f}) → ({fig.bbox.x1:.1f}, {fig.bbox.y1:.1f})")
